# MVRV Exit Deep Dive & Combined Strategy

**Findings so far:**
- MVRV > 3.0 + 20% SL: 62% beat rate (best!)
- But negative avg excess (-8.1%)

**Goals:**
1. Dig into losing periods - why negative excess despite 62% beat rate?
2. Compare MVRV > 2.5 vs 3.0 in detail
3. Create combined strategy: MVRV exit + trailing stop for big runs
4. Find optimal configuration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Deep Dive Time! 🔬")

In [ ]:
# Load all data
DATA_DIR = Path("../data/daily")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")

df = sopr.join(sopr_sth, how='inner').join(price, how='inner').join(mvrv, how='inner')
df = df.sort_index()
df = df[df.index >= '2018-12-15']

close = df['price']
print(f"Data: {len(df)} rows")

In [ ]:
# Entry signal
both_below_1 = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
entries = both_below_1 & ~both_below_1.shift(1).fillna(False)
print(f"Entry signals: {entries.sum()}")

In [ ]:
def backtest_mvrv_exit(df, entries, exit_mvrv=3.0, stop_loss=0.20, max_hold_days=365):
    """Backtest with MVRV exit."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        entry_mvrv = df['mvrv'].iloc[entry_idx]
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            pnl = (current_price - entry_price) / entry_price
            
            if stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if current_mvrv >= exit_mvrv:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'mvrv_exit'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'entry_price': entry_price,
            'entry_mvrv': entry_mvrv,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'exit_mvrv': df.loc[exit_date, 'mvrv'] if exit_date in df.index else np.nan,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

---
## 1. Dig Into Losing Periods

In [ ]:
# Walk-forward with detailed period analysis
def walk_forward_detailed(df, entries, exit_mvrv, stop_loss, train_days=365, test_days=90, step_days=90):
    results = []
    close = df['price']
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end]
        test_entries = entries.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        trades = backtest_mvrv_exit(test_df, test_entries, exit_mvrv, stop_loss)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'fold': fold,
            'start_date': df.index[test_start],
            'end_date': df.index[test_end-1],
            'period': df.index[test_start].strftime('%Y-%m'),
            'n_trades': len(trades),
            'strat_return': strat_return,
            'hold_return': hold_return,
            'excess': strat_return - hold_return,
            'beat_hold': strat_return > hold_return,
            'start_price': test_close.iloc[0],
            'end_price': test_close.iloc[-1],
            'start_mvrv': test_df['mvrv'].iloc[0],
            'end_mvrv': test_df['mvrv'].iloc[-1]
        })
    
    return pd.DataFrame(results)

# Run for MVRV > 3.0
wf_detailed = walk_forward_detailed(df, entries, exit_mvrv=3.0, stop_loss=0.20)

print("WALK-FORWARD PERIOD DETAILS (MVRV > 3.0 + 20% SL)")
print("="*120)
print(f"{'Period':<10} {'Trades':>7} {'Strat':>10} {'B&H':>10} {'Excess':>10} {'Beat':>6} {'MVRV Start':>12} {'MVRV End':>10}")
print("-"*120)

for _, row in wf_detailed.iterrows():
    beat = '✓' if row['beat_hold'] else '✗'
    print(f"{row['period']:<10} {row['n_trades']:>7} {row['strat_return']*100:>9.1f}% {row['hold_return']*100:>9.1f}% "
          f"{row['excess']*100:>+9.1f}% {beat:>6} {row['start_mvrv']:>12.2f} {row['end_mvrv']:>10.2f}")

In [ ]:
# Analyze losing periods
losers = wf_detailed[~wf_detailed['beat_hold']].copy()
winners = wf_detailed[wf_detailed['beat_hold']].copy()

print(f"\n\nLOSING PERIODS ANALYSIS")
print("="*80)
print(f"\nLosing periods: {len(losers)} ({len(losers)/len(wf_detailed)*100:.0f}%)")
print(f"Winning periods: {len(winners)} ({len(winners)/len(wf_detailed)*100:.0f}%)")

print(f"\n{'Metric':<25} {'Winners':>15} {'Losers':>15}")
print("-"*60)
print(f"{'Avg Trades':<25} {winners['n_trades'].mean():>15.1f} {losers['n_trades'].mean():>15.1f}")
print(f"{'Avg Strat Return':<25} {winners['strat_return'].mean()*100:>14.1f}% {losers['strat_return'].mean()*100:>14.1f}%")
print(f"{'Avg B&H Return':<25} {winners['hold_return'].mean()*100:>14.1f}% {losers['hold_return'].mean()*100:>14.1f}%")
print(f"{'Avg Excess':<25} {winners['excess'].mean()*100:>+14.1f}% {losers['excess'].mean()*100:>+14.1f}%")
print(f"{'Avg Start MVRV':<25} {winners['start_mvrv'].mean():>15.2f} {losers['start_mvrv'].mean():>15.2f}")

print(f"\n\nLOSING PERIODS DETAIL:")
print("-"*80)
for _, row in losers.iterrows():
    print(f"{row['period']}: B&H {row['hold_return']*100:+.1f}%, Strat {row['strat_return']*100:+.1f}%, "
          f"Excess {row['excess']*100:+.1f}%, {row['n_trades']} trades")

In [ ]:
# Key insight: When do we lose?
print("\n\nKEY INSIGHT: WHY NEGATIVE AVG EXCESS?")
print("="*80)

# Calculate magnitude of wins vs losses
win_excess = winners['excess'].sum()
lose_excess = losers['excess'].sum()

print(f"\nTotal excess from winning periods: {win_excess*100:+.1f}%")
print(f"Total excess from losing periods: {lose_excess*100:+.1f}%")
print(f"Net: {(win_excess + lose_excess)*100:+.1f}%")

print(f"\nAvg win magnitude: {winners['excess'].mean()*100:+.1f}%")
print(f"Avg loss magnitude: {losers['excess'].mean()*100:.1f}%")

# The problem: big losses in bull markets where B&H does very well
print(f"\n🔍 Biggest losing period excess:")
worst = losers.loc[losers['excess'].idxmin()]
print(f"   {worst['period']}: B&H {worst['hold_return']*100:+.1f}%, Strat {worst['strat_return']*100:+.1f}%")
print(f"   We missed {worst['excess']*100:.1f}% of gains!")

In [ ]:
# Visualize winning vs losing periods
fig = go.Figure()

fig.add_trace(go.Bar(
    x=wf_detailed['period'],
    y=wf_detailed['excess'] * 100,
    marker_color=['green' if x else 'red' for x in wf_detailed['beat_hold']],
    text=[f"{x*100:+.0f}%" for x in wf_detailed['excess']],
    textposition='outside'
))

fig.add_hline(y=0, line_dash='dash', line_color='black')

fig.update_layout(
    title='Excess Return by Period (MVRV > 3.0 + 20% SL)<br><sup>Green = Beat B&H, Red = Lost to B&H</sup>',
    yaxis_title='Excess Return (%)',
    xaxis_tickangle=-45,
    height=500
)
fig.show()

---
## 2. Compare MVRV > 2.5 vs 3.0

In [ ]:
# Get trades for both
trades_25 = backtest_mvrv_exit(df, entries, exit_mvrv=2.5, stop_loss=0.20)
trades_30 = backtest_mvrv_exit(df, entries, exit_mvrv=3.0, stop_loss=0.20)

print("MVRV > 2.5 vs MVRV > 3.0 COMPARISON")
print("="*80)

def summarize(trades, name):
    total_ret = (1 + trades['pnl_pct']).prod() - 1
    win_rate = (trades['pnl_pct'] > 0).mean()
    avg_win = trades[trades['pnl_pct'] > 0]['pnl_pct'].mean() if (trades['pnl_pct'] > 0).any() else 0
    avg_loss = trades[trades['pnl_pct'] <= 0]['pnl_pct'].mean() if (trades['pnl_pct'] <= 0).any() else 0
    avg_days = trades['days_held'].mean()
    
    print(f"\n{name}:")
    print(f"  Trades: {len(trades)}")
    print(f"  Total Return: {total_ret*100:.0f}%")
    print(f"  Win Rate: {win_rate*100:.0f}%")
    print(f"  Avg Win: {avg_win*100:.1f}%")
    print(f"  Avg Loss: {avg_loss*100:.1f}%")
    print(f"  Avg Hold: {avg_days:.0f} days")
    print(f"  Exit breakdown: {trades['exit_reason'].value_counts().to_dict()}")

summarize(trades_25, "MVRV > 2.5 + 20% SL")
summarize(trades_30, "MVRV > 3.0 + 20% SL")

In [ ]:
# Walk-forward comparison
wf_25 = walk_forward_detailed(df, entries, exit_mvrv=2.5, stop_loss=0.20)
wf_30 = walk_forward_detailed(df, entries, exit_mvrv=3.0, stop_loss=0.20)

print("\n\nWALK-FORWARD COMPARISON")
print("="*60)
print(f"{'Metric':<25} {'MVRV > 2.5':>15} {'MVRV > 3.0':>15}")
print("-"*60)
print(f"{'Beat Rate':<25} {wf_25['beat_hold'].mean()*100:>14.0f}% {wf_30['beat_hold'].mean()*100:>14.0f}%")
print(f"{'Avg Excess':<25} {wf_25['excess'].mean()*100:>+14.1f}% {wf_30['excess'].mean()*100:>+14.1f}%")
print(f"{'Avg Trades/Period':<25} {wf_25['n_trades'].mean():>15.1f} {wf_30['n_trades'].mean():>15.1f}")

---
## 3. Combined Strategy: MVRV Exit + Trailing Stop

In [ ]:
def backtest_combined(
    df, entries,
    exit_mvrv=3.0,
    stop_loss=0.20,
    trailing_stop=0.15,
    min_profit_to_trail=0.20,  # Only activate trailing after 20% gain
    max_hold_days=365
):
    """
    Combined strategy:
    - Exit on MVRV threshold (valuation-based)
    - Stop loss for protection
    - Trailing stop to lock in big gains (for parabolic moves that might not hit MVRV 3)
    """
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        entry_mvrv = df['mvrv'].iloc[entry_idx]
        
        peak_price = entry_price
        is_trailing = False
        current_stop = entry_price * (1 - stop_loss) if stop_loss else 0
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            # Update peak
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            pnl_from_peak = (current_price - peak_price) / peak_price
            
            # 1. Check MVRV exit (primary)
            if current_mvrv >= exit_mvrv:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'mvrv_exit'
                break
            
            # 2. Activate trailing stop after min_profit
            if not is_trailing and pnl >= min_profit_to_trail:
                is_trailing = True
            
            # 3. Check trailing stop (if activated)
            if is_trailing and trailing_stop:
                trail_stop_level = peak_price * (1 - trailing_stop)
                if current_price <= trail_stop_level:
                    exit_date = current_date
                    exit_price = trail_stop_level
                    exit_reason = 'trailing_stop'
                    break
            
            # 4. Check initial stop loss
            if stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            # 5. Max hold
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'entry_price': entry_price,
            'entry_mvrv': entry_mvrv,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'exit_mvrv': df.loc[exit_date, 'mvrv'] if exit_date in df.index else np.nan,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason,
            'peak_price': peak_price
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
# Test combined strategies
combined_configs = [
    {'name': 'MVRV>3 only', 'mvrv': 3.0, 'sl': 0.20, 'trail': None, 'min_profit': None},
    {'name': 'MVRV>3 + 15% trail @20%', 'mvrv': 3.0, 'sl': 0.20, 'trail': 0.15, 'min_profit': 0.20},
    {'name': 'MVRV>3 + 20% trail @30%', 'mvrv': 3.0, 'sl': 0.20, 'trail': 0.20, 'min_profit': 0.30},
    {'name': 'MVRV>3 + 25% trail @50%', 'mvrv': 3.0, 'sl': 0.20, 'trail': 0.25, 'min_profit': 0.50},
    {'name': 'MVRV>2.5 + 15% trail @20%', 'mvrv': 2.5, 'sl': 0.20, 'trail': 0.15, 'min_profit': 0.20},
    {'name': 'MVRV>2.5 + 20% trail @30%', 'mvrv': 2.5, 'sl': 0.20, 'trail': 0.20, 'min_profit': 0.30},
]

combined_results = []

for config in combined_configs:
    trades = backtest_combined(
        df, entries,
        exit_mvrv=config['mvrv'],
        stop_loss=config['sl'],
        trailing_stop=config['trail'],
        min_profit_to_trail=config['min_profit'] if config['min_profit'] else 1.0  # Never activate if None
    )
    
    total_ret = (1 + trades['pnl_pct']).prod() - 1
    win_rate = (trades['pnl_pct'] > 0).mean()
    
    combined_results.append({
        'name': config['name'],
        'n_trades': len(trades),
        'total_return': total_ret,
        'win_rate': win_rate,
        'exit_breakdown': trades['exit_reason'].value_counts().to_dict()
    })

print("COMBINED STRATEGY IN-SAMPLE RESULTS")
print("="*100)
print(f"{'Strategy':<30} {'Trades':>8} {'Return':>12} {'Win Rate':>10}")
print("-"*100)
for r in combined_results:
    print(f"{r['name']:<30} {r['n_trades']:>8} {r['total_return']*100:>11.0f}% {r['win_rate']*100:>9.0f}%")
    print(f"{'':>30} Exit breakdown: {r['exit_breakdown']}")

In [ ]:
# Walk-forward for combined strategies
def walk_forward_combined(df, entries, exit_mvrv, stop_loss, trailing_stop, min_profit,
                          train_days=365, test_days=90, step_days=90):
    results = []
    close = df['price']
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end]
        test_entries = entries.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        trades = backtest_combined(
            test_df, test_entries,
            exit_mvrv=exit_mvrv,
            stop_loss=stop_loss,
            trailing_stop=trailing_stop,
            min_profit_to_trail=min_profit if min_profit else 1.0
        )
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'excess': strat_return - hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    return pd.DataFrame(results)

# Walk-forward all combined strategies
wf_combined = []

for config in combined_configs:
    wf = walk_forward_combined(
        df, entries,
        exit_mvrv=config['mvrv'],
        stop_loss=config['sl'],
        trailing_stop=config['trail'],
        min_profit=config['min_profit']
    )
    
    wf_combined.append({
        'strategy': config['name'],
        'beat_rate': wf['beat_hold'].mean(),
        'avg_excess': wf['excess'].mean()
    })

wf_combined_df = pd.DataFrame(wf_combined).sort_values('beat_rate', ascending=False)

print("\n\nWALK-FORWARD: COMBINED STRATEGIES")
print("="*80)
print(f"{'Strategy':<35} {'Beat Rate':>15} {'Avg Excess':>15}")
print("-"*80)
for _, row in wf_combined_df.iterrows():
    print(f"{row['strategy']:<35} {row['beat_rate']*100:>14.0f}% {row['avg_excess']*100:>+14.1f}%")

In [ ]:
# Visualize all strategies
all_strategies = wf_combined_df.copy()

fig = go.Figure()

colors = ['green' if x > 0.60 else 'orange' if x > 0.55 else 'gray' for x in all_strategies['beat_rate']]

fig.add_trace(go.Bar(
    x=all_strategies['strategy'],
    y=all_strategies['beat_rate'] * 100,
    marker_color=colors,
    text=[f"{x:.0f}%" for x in all_strategies['beat_rate']*100],
    textposition='outside'
))

fig.add_hline(y=50, line_dash='dash', line_color='red')
fig.add_hline(y=54, line_dash='dot', line_color='orange', annotation_text='54% baseline')
fig.add_hline(y=62, line_dash='dot', line_color='green', annotation_text='62% MVRV>3')

fig.update_layout(
    title='Walk-Forward Beat Rate - Combined Strategies',
    yaxis_title='Beat Buy & Hold %',
    xaxis_tickangle=-45,
    height=500
)
fig.show()

---
## 4. Final Summary

In [ ]:
print("\n" + "="*80)
print("FINAL STRATEGY COMPARISON")
print("="*80)

# Best strategy
best = wf_combined_df.iloc[0]

print(f"\n🏆 BEST OVERALL: {best['strategy']}")
print(f"   Beat Rate: {best['beat_rate']*100:.0f}%")
print(f"   Avg Excess: {best['avg_excess']*100:+.1f}%")

print(f"\n📊 PROGRESSION:")
print(f"   Baseline (trailing stop only): 54%")
print(f"   MVRV > 3.0 + 20% SL: 62%")
print(f"   Best combined: {best['beat_rate']*100:.0f}%")

print(f"\n💡 KEY INSIGHTS:")
print(f"   1. MVRV > 3.0 is the key exit signal (valuation-based)")
print(f"   2. 20% stop loss protects against bear market entries")
print(f"   3. We beat B&H {best['beat_rate']*100:.0f}% of periods")
print(f"   4. Negative avg excess = we miss some of the biggest runs")
print(f"      (When MVRV doesn't hit 3.0 during a strong bull period)")

print("\n" + "="*80)

In [ ]:
# Show best trades
best_config = combined_configs[[c['name'] for c in combined_configs].index(best['strategy'])]

best_trades = backtest_combined(
    df, entries,
    exit_mvrv=best_config['mvrv'],
    stop_loss=best_config['sl'],
    trailing_stop=best_config['trail'],
    min_profit_to_trail=best_config['min_profit'] if best_config['min_profit'] else 1.0
)

print(f"\n\nBEST STRATEGY TRADES: {best['strategy']}")
print("="*120)

display = best_trades.copy()
display['entry_date'] = pd.to_datetime(display['entry_date']).dt.strftime('%Y-%m-%d')
display['exit_date'] = pd.to_datetime(display['exit_date']).dt.strftime('%Y-%m-%d')
display['entry_price'] = display['entry_price'].round(0).astype(int)
display['exit_price'] = display['exit_price'].round(0).astype(int)
display['pnl_pct'] = (display['pnl_pct'] * 100).round(1)
display['entry_mvrv'] = display['entry_mvrv'].round(2)
display['exit_mvrv'] = display['exit_mvrv'].round(2)

print(display[['entry_date', 'exit_date', 'entry_price', 'exit_price', 'pnl_pct', 'days_held', 'exit_reason', 'entry_mvrv', 'exit_mvrv']].to_string(index=False))

print(f"\nExit Reason Breakdown:")
print(best_trades['exit_reason'].value_counts())

In [ ]:
# Save final results
import json

final_results = {
    'best_strategy': {
        'name': best['strategy'],
        'beat_rate': float(best['beat_rate']),
        'avg_excess': float(best['avg_excess'])
    },
    'strategy_details': {
        'entry': 'SOPR < 1 AND STH_SOPR < 1',
        'exit_mvrv': best_config['mvrv'],
        'stop_loss': best_config['sl'],
        'trailing_stop': best_config['trail'],
        'min_profit_to_trail': best_config['min_profit']
    },
    'all_strategies_tested': wf_combined_df.to_dict('records'),
    'progression': {
        'baseline_trailing_stop': 0.54,
        'mvrv_3_plus_sl': 0.625,
        'best_combined': float(best['beat_rate'])
    }
}

with open('../data/sopr_final_mvrv_strategy.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("Saved to ../data/sopr_final_mvrv_strategy.json")